# SUPP: Why shared human/bovine peptides centre near ratio 1 (ortholog abundance check).

```
SUPP: Why shared human/bovine peptides centre near ratio 1 (ortholog abundance check).

Companion to FIG05. A shared peptide's A/C is set by the abundance of the human (h) and
bovine (b) proteins that contain it:

    A/C = (0.45 h + 0.05 b) / (0.03 h + 0.47 b)       (= 1 only when h = b)

Independent test of "liver orthologs are at similar abundance": estimate h and b from
peptides UNIQUE to each protein (not from the shared peptide), predict the shared
peptide's A/C, and compare with the observed A/C.

    h_hat = median over human-protein-unique peptides of  I_A / 0.45
    b_hat = median over bovine-protein-unique peptides of I_C / 0.47

  Panel A  log2(h/b) per shared peptide (from its ortholog pair).
  Panel B  predicted vs. observed log2(A/C) of the shared peptide.

INCLUDED: shared human+bovine peptides (sequence level, in-silico digest as FIG05) that map
to exactly one human and one bovine protein, quantified in A and C, with >= MIN_UNIQUE
protein-unique peptides on each protein. "Protein-unique" = maps to one protein in the
whole study FASTA (contaminants excluded).

Input: DIA-NN report.pr_matrix.tsv (site 42, same run as FIG02/FIG05); precursors summed
to peptide. No normalization (trout A/C median = 1.00 on this run).
Outputs: output/SUPP_ortholog_abundance.{png,pdf}; data/supp_ortholog_abundance.csv.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA42 = r"D:/2022 Multi-Species Standard Study/42"
INPUT = os.path.join(DATA42, "DIANN_out", "report.pr_matrix.tsv")
FASTA = r"D:/2022 Multi-Species Standard Study/fasta/LakeTrout-Human-Cow-Contaminants-sPRG2022.fasta"
PEPTIDE_COL = "Stripped.Sequence"
COLS = {"A": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_A.mzML",
        "B": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_B.mzML",
        "C": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_C.mzML"}

OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

MIX = {"A": {"Human": 0.45, "Bovine": 0.05, "Trout": 0.50},
       "B": {"Human": 0.20, "Bovine": 0.30, "Trout": 0.50},
       "C": {"Human": 0.03, "Bovine": 0.47, "Trout": 0.50}}
MIN_UNIQUE = 2                      # protein-unique peptides required on each protein
SHARED_COLOR = "#CC79A7"            # human+bovine shared, as in FIG05

In [2]:
def clean_pep(p):
    return re.sub(r"[^A-Z]", "", re.sub(r"\[.*?\]", "", str(p).upper()))


def digest(seq, missed=2, min_len=6, max_len=50):
    seq = re.sub(r"[^A-Z]", "", seq.upper())
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if seq[i] in "KR" and seq[i + 1] != "P"] + [len(seq)]
    cuts = sorted(set(cuts))
    peps = set()
    for i in range(len(cuts) - 1):
        for m in range(missed + 1):
            j = i + 1 + m
            if j >= len(cuts):
                break
            p = seq[cuts[i]:cuts[j]]
            if min_len <= len(p) <= max_len:
                peps.add(p)
    return peps


def fasta_org(tok):
    if "Cont_" in tok:
        return None          # contaminant -- not part of the 3-proteome design
    if "_HUMAN" in tok:
        return "Human"
    if "_BOVIN" in tok:
        return "Bovine"
    return "Trout"           # trout is UniProt (_SALNM) in the study FASTA


def map_proteins(targets):
    """peptide -> list of (non-contaminant) protein accessions containing it; protein -> organism."""
    prot_of = {p: [] for p in targets}
    org_of = {}

    def flush(header, parts):
        if not header:
            return
        tok = header.split()[0]
        org = fasta_org(tok)
        if org is None:
            return
        org_of[tok] = org
        for p in digest("".join(parts)):
            if p in prot_of:
                prot_of[p].append(tok)

    header, parts = None, []
    with open(FASTA, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(header, parts)
                header, parts = line[1:].strip(), []
            else:
                parts.append(line.strip())
        flush(header, parts)
    return prot_of, org_of

In [3]:
def load():
    q = pd.read_csv(INPUT, sep="\t", engine="python", on_bad_lines="skip")
    q.columns = [str(c).strip() for c in q.columns]
    q["peptide"] = q[PEPTIDE_COL].map(clean_pep)
    for k, col in COLS.items():
        q[k] = pd.to_numeric(q[col], errors="coerce")
    print(f"precursor rows: {len(q)}")
    pep = q.groupby("peptide")[["A", "B", "C"]].sum(min_count=1)   # precursors -> peptide
    print(f"peptides: {len(pep)}; dtypes {dict(pep.dtypes.astype(str))}")
    return pep


def build(pep):
    prot_of, org_of = map_proteins(set(pep.index))
    uniq = {}
    for p, prots in prot_of.items():
        if len(prots) == 1:
            uniq.setdefault(prots[0], []).append(p)

    def abundance(prot, sample):
        v = pep.loc[uniq.get(prot, []), sample] / MIX[sample][org_of[prot]]
        v = v[v > 0]
        return (float(np.median(v)) if len(v) else np.nan), int(len(v))

    # trout-only peptides: loading check (no normalization applied)
    trout = [p for p, prots in prot_of.items() if prots and {org_of[t] for t in prots} == {"Trout"}]
    t = pep.loc[trout]; t = t[(t["A"] > 0) & (t["C"] > 0)]
    print(f"trout-only peptides: {len(t)}, median A/C = {2 ** np.median(np.log2(t['A'] / t['C'])):.2f}")

    rows, n_shared, n_multi = [], 0, 0
    for p, prots in prot_of.items():
        if {org_of[t] for t in prots} != {"Human", "Bovine"}:
            continue
        n_shared += 1
        hp = [t for t in prots if org_of[t] == "Human"]
        bp = [t for t in prots if org_of[t] == "Bovine"]
        if len(hp) != 1 or len(bp) != 1:
            n_multi += 1
            continue
        a, c = pep.at[p, "A"], pep.at[p, "C"]
        if not (a > 0 and c > 0):
            continue
        h, nh = abundance(hp[0], "A")
        b, nb = abundance(bp[0], "C")
        rows.append(dict(peptide=p, human_prot=hp[0], bovine_prot=bp[0],
                         obs_log2AC=np.log2(a / c), h_hat=h, n_unique_h=nh, b_hat=b, n_unique_b=nb))
    d = pd.DataFrame(rows)
    print(f"shared human+bovine peptides (sequence): {n_shared}")
    print(f"  excluded, >1 human or >1 bovine protein: {n_multi}")
    print(f"  1:1 ortholog pair, quantified in A and C: {len(d)}")
    d["log2_h_over_b"] = np.log2(d["h_hat"] / d["b_hat"])
    r = d["h_hat"] / d["b_hat"]
    d["pred_log2AC"] = np.log2((MIX["A"]["Human"] * r + MIX["A"]["Bovine"]) /
                               (MIX["C"]["Human"] * r + MIX["C"]["Bovine"]))
    return d


def summarize(x, tag):
    rho = x["obs_log2AC"].rank().corr(x["pred_log2AC"].rank())     # Spearman
    s = dict(subset=tag, n_peptides=len(x), n_human_prot=x["human_prot"].nunique(),
             n_bovine_prot=x["bovine_prot"].nunique(),
             median_log2_h_over_b=x["log2_h_over_b"].median(),
             q25_log2_h_over_b=x["log2_h_over_b"].quantile(.25),
             q75_log2_h_over_b=x["log2_h_over_b"].quantile(.75),
             median_obs_log2AC=x["obs_log2AC"].median(),
             median_pred_log2AC=x["pred_log2AC"].median(),
             spearman=rho, median_abs_err=(x["obs_log2AC"] - x["pred_log2AC"]).abs().median())
    return s

In [4]:
def figure(x, s, out_png):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.6))
    ax1.hist(x["log2_h_over_b"], bins=60, color=SHARED_COLOR)
    ax1.axvline(0, ls="--", lw=1.2, color="0.3", label="equal abundance (h = b)")
    ax1.axvline(s["median_log2_h_over_b"], ls="-", lw=1.6, color="black",
                label=f"median = {s['median_log2_h_over_b']:+.2f}")
    ax1.set_xlabel("log2(human / bovine ortholog abundance)\nfrom protein-unique peptides")
    ax1.set_ylabel("shared peptides (count)")
    ax1.set_title("A   Ortholog abundance ratio", loc="left", fontsize=11)
    ax1.legend(frameon=False, fontsize=9, loc="upper left")

    lim = [-3.5, 4.2]
    ax2.scatter(x["pred_log2AC"], x["obs_log2AC"], s=6, alpha=0.4, color=SHARED_COLOR, linewidth=0)
    ax2.plot(lim, lim, ls="--", lw=1, color="0.3", label="observed = predicted")
    ax2.set_xlim(lim); ax2.set_ylim(lim); ax2.grid(True, lw=0.3)
    ax2.set_xlabel("predicted log2(A/C) of shared peptide")
    ax2.set_ylabel("observed log2(A/C) of shared peptide")
    ax2.set_title("B   Predicted vs. observed", loc="left", fontsize=11)
    ax2.text(0.97, 0.04, f"median predicted {s['median_pred_log2AC']:+.2f}\n"
                         f"median observed {s['median_obs_log2AC']:+.2f}\n"
                         f"Spearman rho = {s['spearman']:.2f}",
             transform=ax2.transAxes, ha="right", va="bottom", fontsize=9)
    ax2.legend(frameon=False, fontsize=9, loc="upper left")

    fig.suptitle(f"SUPP  Shared human/bovine peptide centre near ratio 1 is consistent with similar "
                 f"liver ortholog abundance\n(site 42, DIA-NN; 1:1 ortholog pairs, >= {MIN_UNIQUE} "
                 f"protein-unique peptides each; n = {s['n_peptides']:,} peptides)", fontsize=11)
    fig.subplots_adjust(left=0.07, right=0.98, top=0.80, bottom=0.17, wspace=0.25)
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_png.replace(".png", ".pdf"))
    plt.close(fig)
    print(f"  wrote {out_png}")

In [5]:
# ---- generate figure ----
pep = load()
d = build(pep)
d.to_csv(os.path.join(DATA, "supp_ortholog_abundance.csv"), index=False, lineterminator="\n")
any1 = d.dropna(subset=["h_hat", "b_hat"])
strict = any1[(any1["n_unique_h"] >= MIN_UNIQUE) & (any1["n_unique_b"] >= MIN_UNIQUE)]
summ = pd.DataFrame([summarize(any1, ">=1 unique peptide each"),
                     summarize(strict, f">={MIN_UNIQUE} unique peptides each")])
with pd.option_context("display.width", 200, "display.float_format", "{:+.2f}".format):
    print(summ.to_string(index=False))
figure(strict, summ.iloc[1].to_dict(), os.path.join(OUTPUT, "SUPP_ortholog_abundance.png"))

precursor rows: 25229
peptides: 20829; dtypes {'A': 'float64', 'B': 'float64', 'C': 'float64'}


trout-only peptides: 2767, median A/C = 1.00


shared human+bovine peptides (sequence): 5084
  excluded, >1 human or >1 bovine protein: 1638
  1:1 ortholog pair, quantified in A and C: 2838
                  subset  n_peptides  n_human_prot  n_bovine_prot  median_log2_h_over_b  q25_log2_h_over_b  q75_log2_h_over_b  median_obs_log2AC  median_pred_log2AC  spearman  median_abs_err
 >=1 unique peptide each        1613           383            383                 -0.27              -0.98              +0.54              -0.18               -0.23     +0.37           +0.69
>=2 unique peptides each        1316           277            278                 -0.25              -0.95              +0.44              -0.18               -0.21     +0.37           +0.68


  wrote output\SUPP_ortholog_abundance.png
